In [1]:
# 02_resnet_bottleneck_flops.py
import torch
import torch.nn as nn

class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False)
        )
    def forward(self, x): return self.net(x)

class BottleneckBlock(nn.Module):
    def __init__(self, in_ch, mid_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, mid_ch, kernel_size=1, bias=False),  # 차원 축소
            nn.Conv2d(mid_ch, mid_ch, kernel_size=3, padding=1, bias=False),
            nn.Conv2d(mid_ch, out_ch, kernel_size=1, bias=False)  # 차원 확장
        )
    def forward(self, x): return self.net(x)

# 대형 레이어 상태 가정 (입력 1024 채널 -> 출력 1024 채널)
in_channels = 1024
out_channels = 1024

basic = BasicBlock(in_channels, out_channels)
bottleneck = BottleneckBlock(in_channels, mid_ch=256, out_ch=out_channels)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

print("=== [실험 3] 대용량 채널 구간에서 두 블록의 효율성 대조 ===")
print(f"Basic Block 파라미터 총합: {count_params(basic):,} 개")
print(f"Bottleneck Block 파라미터 총합: {count_params(bottleneck):,} 개")
print(f"-> Bottleneck 채택 시 파라미터 약 {count_params(basic)/count_params(bottleneck):.1f}배 절감 효과 증명.")

=== [실험 3] 대용량 채널 구간에서 두 블록의 효율성 대조 ===
Basic Block 파라미터 총합: 18,874,368 개
Bottleneck Block 파라미터 총합: 1,114,112 개
-> Bottleneck 채택 시 파라미터 약 16.9배 절감 효과 증명.
